[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/02_sizing_and_serving/02.1_deploying_your_model/lab.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/02_sizing_and_serving/02.1_deploying_your_model/lab.ipynb)

# Lab 2.1: Capacity Planning and GPU Selection

Interactive deployment calculator.

In [6]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML

# Model specifications: params in billions, layer/head/dim counts
MODELS = {
    'Mistral-7B':  {'params_B': 7.24,  'layers': 32,  'heads': 32,  'kv_heads': 8, 'head_dim': 128},
    'Llama-70B':   {'params_B': 70.6,  'layers': 80,  'heads': 64,  'kv_heads': 8, 'head_dim': 128},
    'Llama-405B':  {'params_B': 405.0, 'layers': 126, 'heads': 128,'kv_heads': 8, 'head_dim': 128},
}

# GPU VRAM in GB
GPUS = {
    'T4': 16, 'A10G': 24, 'A100-80': 80, 'H100': 80, 'H200': 141,
}

# Bytes per parameter for each precision
PRECISIONS = {
    'FP16': 2, 'INT8': 1, 'INT4': 0.5,
}

# Approximate cost per hour (USD) for reference
GPU_COST = {
    'T4': 0.53, 'A10G': 1.21, 'A100-80': 3.67, 'H100': 4.25, 'H200': 5.50,
}

print('Models:', list(MODELS.keys()))
print('GPUs:', list(GPUS.keys()))
print('Precisions:', list(PRECISIONS.keys()))

# --- Batch analysis (A100-80, Mistral-7B FP16) ---

# GPU and model parameters (A100 80GB)
bandwidth_gbs = 2039        # HBM bandwidth in GB/s
model_size_gb = 14.5        # Model weights in GB (7B params in fp16)
kv_per_token_mb = 2         # KV cache per token in MB

# Batch sizes to evaluate
batch_sizes = np.array([1, 2, 4, 8, 16, 32, 64, 128])

# --- Instance catalog ---

# GPU instance catalog: real cloud offerings
GPU_INSTANCES = {
    "A10G (24GB)": {"vram_gb": 24, "bandwidth_gbs": 600, "flops_tflops": 31.2, "cost_per_hr": 1.01},
    "L4 (24GB)": {"vram_gb": 24, "bandwidth_gbs": 300, "flops_tflops": 30.3, "cost_per_hr": 0.81},
    "A100 40GB": {"vram_gb": 40, "bandwidth_gbs": 1555, "flops_tflops": 77.9, "cost_per_hr": 3.67},
    "A100 80GB": {"vram_gb": 80, "bandwidth_gbs": 2039, "flops_tflops": 77.9, "cost_per_hr": 5.12},
    "H100 80GB": {"vram_gb": 80, "bandwidth_gbs": 3350, "flops_tflops": 267.6, "cost_per_hr": 8.10},
    "L40S (48GB)": {"vram_gb": 48, "bandwidth_gbs": 864, "flops_tflops": 91.6, "cost_per_hr": 2.40},
    "H200 (141GB)": {"vram_gb": 141, "bandwidth_gbs": 4800, "flops_tflops": 267.6, "cost_per_hr": 12.50},
    "MI300X (192GB)": {"vram_gb": 192, "bandwidth_gbs": 5300, "flops_tflops": 163.4, "cost_per_hr": 10.00},
}

# Bytes per parameter by precision
PRECISION_BYTES = {"FP16": 2, "INT8": 1, "INT4": 0.5}

# Model sizes in billions of parameters
MODEL_PARAMS = {"7B": 7, "13B": 13, "70B": 70, "405B": 405}


Models: ['Mistral-7B', 'Llama-70B', 'Llama-405B']
GPUs: ['T4', 'A10G', 'A100-80', 'H100', 'H200']
Precisions: ['FP16', 'INT8', 'INT4']


## Step-by-Step Capacity Calculator

In [7]:
def capacity_calc(model_name, gpu_name, precision, tokens_per_conversation, num_users):
    """Show every calculation step for GPU memory capacity planning."""
    m = MODELS[model_name]
    gpu_vram = GPUS[gpu_name]
    bpp = PRECISIONS[precision]  # bytes per parameter

    # Step 1: Weight memory
    weight_gb = m['params_B'] * bpp  # billions of params * bytes = GB
    print(f'Step 1: Weight Memory')
    print(f'  params x bytes_per_param = {m["params_B"]}B x {bpp} = {weight_gb:.1f} GB')
    print()

    # Step 2: KV cache per token (bytes, then convert)
    # Formula: 2 (K+V) x heads x head_dim x layers x bytes_per_value
    kv_bytes_per_token = 2 * m.get('kv_heads', m['heads']) * m['head_dim'] * m['layers'] * bpp
    kv_mb_per_token = kv_bytes_per_token / (1024**2)
    print(f'Step 2: KV Cache Per Token')
    print(f'  2 (K+V) x {m["heads"]} heads x {m["head_dim"]} dim x {m["layers"]} layers x {bpp} bytes')
    print(f'  = {kv_bytes_per_token:,.0f} bytes = {kv_mb_per_token:.2f} MB/token')
    print()

    # Step 3: KV cache per user
    kv_per_user_gb = (kv_bytes_per_token * tokens_per_conversation) / (1024**3)
    print(f'Step 3: KV Cache Per User')
    print(f'  {kv_mb_per_token:.2f} MB/token x {tokens_per_conversation} tokens = {kv_per_user_gb:.2f} GB/user')
    print()

    # Step 4: Total KV cache for all users
    total_kv_gb = kv_per_user_gb * num_users
    print(f'Step 4: Total KV Cache')
    print(f'  {kv_per_user_gb:.2f} GB/user x {num_users} users = {total_kv_gb:.1f} GB')
    print()

    # Step 5: Total memory (weights + KV + 10% overhead)
    overhead_gb = weight_gb * 0.10  # 10% of weights for activations/framework
    total_gb = weight_gb + total_kv_gb + overhead_gb
    print(f'Step 5: Total Memory Needed')
    print(f'  weights + total_kv + overhead(10%) = {weight_gb:.1f} + {total_kv_gb:.1f} + {overhead_gb:.1f} = {total_gb:.1f} GB')
    print()

    # Step 6: Fit check
    fits = total_gb <= gpu_vram
    headroom = gpu_vram - total_gb
    print(f'Step 6: Does it fit?')
    print(f'  GPU VRAM = {gpu_vram} GB')
    print(f'  Needed   = {total_gb:.1f} GB')
    if fits:
        print(f'  \U00002705 FITS ({headroom:.1f} GB headroom)')
    else:
        print(f'  \U0000274C OOM (need {-headroom:.1f} more GB)')
    print()

    # Stacked bar chart: weights | KV | overhead vs GPU VRAM
    fig, ax = plt.subplots(figsize=(8, 2))
    ax.barh(['Memory'], [weight_gb], color='#dbeafe', edgecolor='#000', label='Weights')
    ax.barh(['Memory'], [total_kv_gb], left=[weight_gb], color='#fef3c7', edgecolor='#000', label='KV Cache')
    ax.barh(['Memory'], [overhead_gb], left=[weight_gb + total_kv_gb], color='#f3e8ff', edgecolor='#000', label='Overhead')
    # GPU VRAM line
    ax.axvline(gpu_vram, color='red', linestyle='--', linewidth=2, label=f'{gpu_name} VRAM ({gpu_vram} GB)')
    ax.set_xlabel('GB')
    ax.set_title('Memory Breakdown vs GPU Capacity')
    ax.legend(loc='upper right', fontsize=8)
    ax.set_xlim(0, max(total_gb, gpu_vram) * 1.1)
    plt.tight_layout()
    plt.show()

# Build interactive widgets
widgets.interact(
    capacity_calc,
    model_name=widgets.Dropdown(options=list(MODELS.keys()), value='Mistral-7B', description='Model:'),
    gpu_name=widgets.Dropdown(options=list(GPUS.keys()), value='A100-80', description='GPU:'),
    precision=widgets.Dropdown(options=list(PRECISIONS.keys()), value='FP16', description='Precision:'),
    tokens_per_conversation=widgets.IntSlider(min=256, max=32768, step=256, value=2048, description='Tokens/conv:',
                                              style={'description_width': 'initial'}),
    num_users=widgets.IntSlider(min=1, max=256, step=1, value=16, description='Concurrent users:',
                                style={'description_width': 'initial'}),
);

interactive(children=(Dropdown(description='Model:', options=('Mistral-7B', 'Llama-70B', 'Llama-405B'), value=…

## Interactive Instance Selector

In [8]:
# Model selection widgets
model_dd = widgets.Dropdown(options=["7B", "13B", "70B", "405B"], value="7B", description="Model:")
prec_dd = widgets.Dropdown(options=["FP16", "INT8", "INT4"], value="FP16", description="Precision:")
ctx_slider = widgets.IntSlider(value=4096, min=512, max=32768, step=512, description="Context:")
batch_slider = widgets.IntSlider(value=8, min=1, max=128, step=1, description="Batch Size:")
itl_slider = widgets.FloatSlider(value=50.0, min=5.0, max=200.0, step=5.0, description="Max ITL (ms):")
out = widgets.Output()

def evaluate_instances(model, precision, tokens_per_conversation, batch_size, max_itl_ms):
    """Check each GPU against model requirements."""
    params_b = MODEL_PARAMS[model]
    bytes_per_param = PRECISION_BYTES[precision]
    # Model weight memory in GB
    weight_gb = params_b * bytes_per_param
    # KV cache per token per layer: 2 * hidden * 2 bytes (FP16 KV always)
    # Approximate: hidden ~ params_b * 1024 / 7 for scaling, layers ~ params_b * 32 / 7
    hidden = int(params_b * 1024 / 7) * 7  # rough scaling
    layers = int(params_b * 32 / 7)
    kv_per_token_gb = 2 * hidden * 2 * layers / 1e9  # 2 matrices, 2 bytes each
    kv_total_gb = kv_per_token_gb * tokens_per_conversation * batch_size
    total_vram_gb = weight_gb + kv_total_gb
    # ITL estimate: weight_gb * 1e9 / bandwidth_bytes_per_sec * 1000 (ms)
    results = []
    for name, spec in GPU_INSTANCES.items():
        vram_ok = total_vram_gb <= spec["vram_gb"] * 0.9  # 90% usable
        # Min ITL: time to read all weights once per token
        min_itl_ms = (weight_gb * 1e3) / spec["bandwidth_gbs"]  # GB / (GB/s) = seconds, *1e3 = ms
        itl_ok = min_itl_ms <= max_itl_ms
        passes = vram_ok and itl_ok
        results.append({"name": name, "vram_ok": vram_ok, "itl_ok": itl_ok,
                        "passes": passes, "min_itl_ms": min_itl_ms,
                        "total_vram_gb": total_vram_gb, "cost": spec["cost_per_hr"]})
    return sorted(results, key=lambda x: (not x["passes"], x["cost"]))

def on_change(_):
    out.clear_output()
    with out:
        results = evaluate_instances(model_dd.value, prec_dd.value,
                                     ctx_slider.value, batch_slider.value, itl_slider.value)
        # Build HTML table
        html = "<table style='border-collapse:collapse;width:100%'>"
        html += "<tr style='background:#f3f4f6'><th>Instance</th><th>VRAM</th><th>ITL</th><th>Cost/hr</th><th>Status</th></tr>"
        for r in results:
            color = "#dcfce7" if r["passes"] else "#ffe4e6"  # green pass, rose fail
            status = "\u2705 PASS" if r["passes"] else "\u274c FAIL"
            html += f"<tr style='background:{color}'><td>{r['name']}</td><td>{r['total_vram_gb']:.1f}/{GPU_INSTANCES[r['name']]['vram_gb']}GB</td>"
            html += f"<td>{r['min_itl_ms']:.1f}ms</td><td>${r['cost']:.2f}</td><td>{status}</td></tr>"
        html += "</table>"
        display(HTML(html))

# Connect widgets
for w in [model_dd, prec_dd, ctx_slider, batch_slider, itl_slider]:
    w.observe(on_change, names='value')

display(widgets.VBox([model_dd, prec_dd, ctx_slider, batch_slider, itl_slider, out]))
on_change(None)  # Initial render

## GPU Selection & Recommendation
Choose model, precision, context, batch, and latency SLO. See all GPUs ranked by cost with the cheapest highlighted.

In [9]:
# GPU Selection: interactive cost comparison + recommendation
# Uses same formulas as capacity calculator above (correct KV math)

sel_model = widgets.Dropdown(options=list(MODELS.keys()), value="Mistral-7B", description="Model:")
sel_prec = widgets.Dropdown(options=list(PRECISIONS.keys()), value="FP16", description="Precision:")
sel_ctx = widgets.IntSlider(value=2048, min=512, max=16384, step=512, description="Context:", style={"description_width": "initial"})
sel_batch = widgets.IntSlider(value=16, min=1, max=128, step=1, description="Batch Size:", style={"description_width": "initial"})
sel_itl = widgets.FloatSlider(value=30.0, min=5.0, max=100.0, step=5.0, description="Max ITL (ms):", style={"description_width": "initial"})
sel_out = widgets.Output()

def evaluate_and_recommend(_):
    sel_out.clear_output(wait=True)
    with sel_out:
        m = MODELS[sel_model.value]
        bpp = PRECISIONS[sel_prec.value]
        tokens = sel_ctx.value
        batch_size = sel_batch.value
        max_itl = sel_itl.value

        # Correct formulas (same as capacity_calc)
        weight_gb = m['params_B'] * bpp
        kv_bytes_per_token = 2 * m.get('kv_heads', m['heads']) * m['head_dim'] * m['layers'] * bpp
        kv_per_user_gb = (kv_bytes_per_token * tokens) / (1024**3)
        overhead_gb = weight_gb * 0.10

        print(f"Model: {sel_model.value} ({sel_prec.value})")
        print(f"Weights: {weight_gb:.1f} GB | KV/user: {kv_per_user_gb*1000:.0f} MB | Batch: {batch_size} | SLO: {max_itl}ms")
        print()

        # Evaluate each GPU
        results = []
        for gn, vram in GPUS.items():
            bw = {'T4': 320, 'A10G': 600, 'A100-80': 2039, 'H100': 3350, 'H200': 4800}[gn]
            cost_hr = GPU_COST[gn]
            available = vram - weight_gb - overhead_gb
            total_kv = kv_per_user_gb * batch_size
            fits = (weight_gb + total_kv + overhead_gb) <= vram

            # ITL = (weights + batch * kv_per_user) / bandwidth * 1000
            itl_ms = (weight_gb + batch_size * kv_per_user_gb) / bw * 1000
            passes_itl = itl_ms <= max_itl

            # Throughput = batch / (itl / 1000)
            tok_per_sec = batch_size / (itl_ms / 1000) if itl_ms > 0 else 0
            cost_per_m = (cost_hr / tok_per_sec) * (1e6 / 3600) if tok_per_sec > 0 else 999

            passes = fits and passes_itl
            results.append({
                "name": gn, "vram": vram, "fits": fits, "itl_ms": itl_ms,
                "passes_itl": passes_itl, "passes": passes,
                "tok_per_sec": tok_per_sec, "cost_per_m": cost_per_m,
                "cost_hr": cost_hr
            })

        # Sort: passing first (by cost), then failing
        results.sort(key=lambda x: (not x["passes"], x["cost_per_m"]))

        # HTML table: all GPUs, green = pass, rose = fail
        html = "<table style='border-collapse:collapse;width:100%;font-size:13px'>"
        html += "<tr style='background:#f3f4f6'><th>GPU</th><th>VRAM</th><th>Fits?</th><th>ITL</th><th>Throughput</th><th>$/hr</th><th>$/M tok</th><th>Status</th></tr>"
        for r in results:
            color = "#dcfce7" if r["passes"] else "#ffe4e6"
            status = "PASS" if r["passes"] else ("OOM" if not r["fits"] else "OVER SLO")
            cpm_str = f"" if r["passes"] else "--"
            html += f"<tr style='background:{color}'><td><b>{r['name']}</b></td><td>{r['vram']} GB</td>"
            html += f"<td>{'Y' if r['fits'] else 'N'}</td><td>{r['itl_ms']:.1f}ms</td>"
            html += f"<td>{r['tok_per_sec']:,.0f} tok/s</td><td></td>"
            html += f"<td>{cpm_str}</td><td><b>{status}</b></td></tr>"
        html += "</table>"
        display(HTML(html))
        print()

        # Cost chart
        passing = [r for r in results if r["passes"]]
        if passing:
            names = [r["name"] for r in passing]
            costs = [r["cost_per_m"] for r in passing]
            cheapest_idx = costs.index(min(costs))
            colors = ["#86efac" if i == cheapest_idx else "#dbeafe" for i in range(len(names))]

            fig, ax = plt.subplots(figsize=(9, max(2.5, len(names)*0.7)))
            ax.barh(names, costs, color=colors, edgecolor="#000", height=0.5)
            for i, (n, c) in enumerate(zip(names, costs)):
                r = passing[i]
                label = f" ${c:.2f}/M | {r['tok_per_sec']:,.0f} tok/s | ITL={r['itl_ms']:.1f}ms"
                if i == cheapest_idx:
                    label += " << CHEAPEST"
                ax.text(c + max(costs)*0.02, i, label, va="center", fontsize=9)
            ax.set_xlabel("$/Million Tokens (lower = better)")
            ax.set_title(f"{sel_model.value} ({sel_prec.value}) | batch={batch_size} | SLO={max_itl}ms")
            ax.invert_yaxis()
            ax.set_xlim(0, max(costs) * 2.0)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)
            plt.tight_layout()
            plt.show()

            # Top recommendation
            best = passing[0]
            print(f"\nRECOMMENDATION: {best['name']}")
            print(f"  ${best['cost_per_m']:.2f}/M tokens | ${best['cost_hr']:.2f}/hr")
            print(f"  {best['tok_per_sec']:,.0f} tok/s | ITL: {best['itl_ms']:.1f}ms")
        else:
            print("No GPU meets these constraints.")
            print("Try: lower precision, shorter context, smaller batch, or higher SLO.")

        # Show failing GPUs
        failing = [r for r in results if not r["passes"]]
        if failing:
            print(f"\nFailing GPUs:")
            for r in failing:
                reason = "OOM" if not r["fits"] else f"ITL={r['itl_ms']:.1f}ms > {max_itl}ms"
                print(f"  {r['name']}: {reason}")

for w in [sel_model, sel_prec, sel_ctx, sel_batch, sel_itl]:
    w.observe(evaluate_and_recommend, names="value")

display(widgets.VBox([sel_model, sel_prec, sel_ctx, sel_batch, sel_itl, sel_out]))
evaluate_and_recommend(None)


## Key Takeaway
**KV cache is the variable cost.** The model weights are fixed; KV cache grows linearly with sequence length and batch size. Your deployment decision comes down to: which GPU gives you enough VRAM and bandwidth at the lowest $/M tokens.